# Air Quality Data Cleaning and Preprocessing

This notebook performs data validation, cleaning, and preprocessing of the raw Delhi air-quality dataset collected from the Open-Meteo Air Quality API.

The raw dataset is preserved in `data/raw/` and will not be modified directly.

### Objectives

- Load the raw air-quality dataset
- Validate data types and timestamps
- Check missing values and duplicate records
- Validate pollutant values
- Check hourly timestamp continuity
- Sort the dataset chronologically
- Create basic time-based features
- Perform a final data-quality check
- Save the cleaned dataset to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
raw_path = Path("../data/raw/air_quality_delhi.csv")

df = pd.read_csv(raw_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (8760, 7)


,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
0,2025-01-01 00:00:00,185.8,188.6,1645.0,56.7,49.7,31.0
1,2025-01-01 01:00:00,174.6,177.4,1551.0,44.8,39.7,39.0
2,2025-01-01 02:00:00,164.4,166.7,1478.0,34.6,31.5,46.0
3,2025-01-01 03:00:00,156.5,158.8,1418.0,26.7,26.0,52.0
4,2025-01-01 04:00:00,149.5,151.8,1379.0,20.6,22.3,57.0


In [3]:
cleaned_df = df.copy()

In [4]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   time              8760 non-null   str    
 1   pm2_5             8760 non-null   float64
 2   pm10              8760 non-null   float64
 3   carbon_monoxide   8760 non-null   float64
 4   nitrogen_dioxide  8760 non-null   float64
 5   sulphur_dioxide   8760 non-null   float64
 6   ozone             8760 non-null   float64
dtypes: float64(6), str(1)
memory usage: 479.2 KB


In [5]:
cleaned_df["time"] = pd.to_datetime(
    cleaned_df["time"],
    errors="coerce"
)

cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   time              8760 non-null   datetime64[us]
 1   pm2_5             8760 non-null   float64       
 2   pm10              8760 non-null   float64       
 3   carbon_monoxide   8760 non-null   float64       
 4   nitrogen_dioxide  8760 non-null   float64       
 5   sulphur_dioxide   8760 non-null   float64       
 6   ozone             8760 non-null   float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 479.2 KB


In [6]:
invalid_timestamps = cleaned_df["time"].isna().sum()

print("Invalid timestamps:", invalid_timestamps)

Invalid timestamps: 0


In [7]:
missing_values = cleaned_df.isnull().sum()

print(missing_values)

print(
    "\nTotal missing values:",
    cleaned_df.isnull().sum().sum()
)

time                0
pm2_5               0
pm10                0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
dtype: int64

Total missing values: 0


In [8]:
duplicate_rows = cleaned_df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


In [9]:
duplicate_timestamps = cleaned_df["time"].duplicated().sum()

print("Duplicate timestamps:", duplicate_timestamps)

Duplicate timestamps: 0


In [10]:
cleaned_df = (
    cleaned_df
    .sort_values("time")
    .reset_index(drop=True)
)

cleaned_df.head()

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
0,2025-01-01 00:00:00,185.8,188.6,1645.0,56.7,49.7,31.0
1,2025-01-01 01:00:00,174.6,177.4,1551.0,44.8,39.7,39.0
2,2025-01-01 02:00:00,164.4,166.7,1478.0,34.6,31.5,46.0
3,2025-01-01 03:00:00,156.5,158.8,1418.0,26.7,26.0,52.0
4,2025-01-01 04:00:00,149.5,151.8,1379.0,20.6,22.3,57.0


In [11]:
pollutants = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

In [12]:
cleaned_df[pollutants].dtypes

pm2_5               float64
pm10                float64
carbon_monoxide     float64
nitrogen_dioxide    float64
sulphur_dioxide     float64
ozone               float64
dtype: object

In [13]:
for column in pollutants:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

In [14]:
print(
    "Missing values after numeric conversion:",
    cleaned_df[pollutants].isnull().sum().sum()
)

Missing values after numeric conversion: 0


In [15]:
negative_values = {}

for column in pollutants:
    count = (cleaned_df[column] < 0).sum()
    negative_values[column] = count

negative_values

{'pm2_5': np.int64(0),
 'pm10': np.int64(0),
 'carbon_monoxide': np.int64(0),
 'nitrogen_dioxide': np.int64(0),
 'sulphur_dioxide': np.int64(0),
 'ozone': np.int64(0)}

In [16]:
invalid_pollutant_records = cleaned_df[
    (cleaned_df[pollutants] < 0).any(axis=1)
]

print(
    "Records containing negative pollutant values:",
    len(invalid_pollutant_records)
)

Records containing negative pollutant values: 0


In [17]:
cleaned_df[pollutants].agg(
    ["min", "max", "mean", "median"]
).T

,min,max,mean,median
pm2_5,6.0,369.7,80.291941,70.5
pm10,6.1,1438.0,199.150742,136.9
carbon_monoxide,176.0,6889.0,864.171575,675.0
nitrogen_dioxide,2.5,180.2,33.837283,26.6
sulphur_dioxide,8.1,185.8,30.627854,26.1
ozone,0.0,331.0,97.560502,82.0


In [18]:
time_difference = cleaned_df["time"].diff()

expected_interval = pd.Timedelta(hours=1)

unexpected_intervals = (
    time_difference.iloc[1:] != expected_interval
)

print(
    "Unexpected hourly intervals:",
    unexpected_intervals.sum()
)

Unexpected hourly intervals: 0


In [19]:
print("Start:", cleaned_df["time"].min())
print("End:", cleaned_df["time"].max())

print("Total observations:", len(cleaned_df))
print("Unique timestamps:", cleaned_df["time"].nunique())

Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00
Total observations: 8760
Unique timestamps: 8760


In [20]:
cleaned_df["year"] = cleaned_df["time"].dt.year
cleaned_df["month"] = cleaned_df["time"].dt.month
cleaned_df["day"] = cleaned_df["time"].dt.day
cleaned_df["hour"] = cleaned_df["time"].dt.hour
cleaned_df["day_of_year"] = cleaned_df["time"].dt.dayofyear

In [21]:
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"


cleaned_df["season"] = cleaned_df["month"].apply(
    assign_season
)

In [22]:
cleaned_df["season"].value_counts()

season
Monsoon         2928
Summer          2208
Winter          2160
Post-Monsoon    1464
Name: count, dtype: int64

In [23]:
cleaned_df.head()

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,year,month,day,hour,day_of_year,season
0,2025-01-01 00:00:00,185.8,188.6,1645.0,56.7,49.7,31.0,2025,1,1,0,1,Winter
1,2025-01-01 01:00:00,174.6,177.4,1551.0,44.8,39.7,39.0,2025,1,1,1,1,Winter
2,2025-01-01 02:00:00,164.4,166.7,1478.0,34.6,31.5,46.0,2025,1,1,2,1,Winter
3,2025-01-01 03:00:00,156.5,158.8,1418.0,26.7,26.0,52.0,2025,1,1,3,1,Winter
4,2025-01-01 04:00:00,149.5,151.8,1379.0,20.6,22.3,57.0,2025,1,1,4,1,Winter


In [24]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   time              8760 non-null   datetime64[us]
 1   pm2_5             8760 non-null   float64       
 2   pm10              8760 non-null   float64       
 3   carbon_monoxide   8760 non-null   float64       
 4   nitrogen_dioxide  8760 non-null   float64       
 5   sulphur_dioxide   8760 non-null   float64       
 6   ozone             8760 non-null   float64       
 7   year              8760 non-null   int32         
 8   month             8760 non-null   int32         
 9   day               8760 non-null   int32         
 10  hour              8760 non-null   int32         
 11  day_of_year       8760 non-null   int32         
 12  season            8760 non-null   str           
dtypes: datetime64[us](1), float64(6), int32(5), str(1)
memory usage: 718.7 KB


In [25]:
print("========== FINAL DATA QUALITY CHECK ==========")

print("\nDataset shape:")
print(cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(cleaned_df.duplicated().sum())

print("\nDuplicate timestamps:")
print(cleaned_df["time"].duplicated().sum())

print("\nUnexpected hourly intervals:")
time_difference = cleaned_df["time"].diff()

unexpected_intervals = (
    time_difference.iloc[1:] != pd.Timedelta(hours=1)
)

print(unexpected_intervals.sum())

print("\nNegative pollutant values:")

for column in pollutants:
    print(
        f"{column}:",
        (cleaned_df[column] < 0).sum()
    )

========== FINAL DATA QUALITY CHECK ==========

Dataset shape:
(8760, 13)

Missing values:
0

Duplicate rows:
0

Duplicate timestamps:
0

Unexpected hourly intervals:
0

Negative pollutant values:
pm2_5: 0
pm10: 0
carbon_monoxide: 0
nitrogen_dioxide: 0
sulphur_dioxide: 0
ozone: 0


In [26]:
processed_path = Path(
    "../data/processed/air_quality_delhi_cleaned.csv"
)

processed_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_df.to_csv(
    processed_path,
    index=False
)

print(
    f"Processed dataset saved to: {processed_path}"
)

Processed dataset saved to: ..\data\processed\air_quality_delhi_cleaned.csv
